# Misclassified pairs — the records behind every error cell

`end_to_end_eval.ipynb` counts what each stage got wrong. This notebook shows
**which pairs**, in the same section order, with the two records stacked and
colour-coded field by field — the `gold_labeling.ipynb` layout — so you can
judge the error rather than just count it.

> ⚠️ **Every output below is PHI.** The two notebooks are separate for exactly
> this reason: the counts one is safe to commit, this one's outputs are not.
> Clear outputs before saving.

Both notebooks are driven by the same `src.evaluation.stage_diagnostics` frame,
so a cell in a matrix there and the rows listed here are always the same pairs —
they cannot drift apart.

## 0. Setup

In [ ]:
import importlib
import sys
from pathlib import Path


def _service_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "data").is_dir() and (d / "src").is_dir():
            return d
    raise FileNotFoundError("could not locate empi-service/ (no ancestor has data/ + src/)")


SERVICE_ROOT = _service_root(Path.cwd())
if str(SERVICE_ROOT) not in sys.path:
    sys.path.insert(0, str(SERVICE_ROOT))

import pandas as pd

from src.config import settings
from src.evaluation.holdout import DEFAULT_GOLD_LABELS
from src.evaluation.report_io import load_reports, summary_frame

# These notebooks are edited alongside src/evaluation/, so a kernel holding an
# older import would shadow the change with a confusing AttributeError. Reloading
# here means re-running this cell is enough — no kernel restart.
import src.evaluation.stage_diagnostics
importlib.reload(src.evaluation.stage_diagnostics)
from src.evaluation import stage_diagnostics as sd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 45)

## 1. Pick the run

`FOCUS` indexes the same stored-report list `end_to_end_eval.ipynb` uses, so
setting it to the same value here lines this notebook up with the matrices
there — same run, same label file, same holdout, same population.

In [ ]:
reports = load_reports()
summary_frame(reports)

In [ ]:
FOCUS = 0
LABELS = DEFAULT_GOLD_LABELS   # point at the synthetic/silver file for those reports

report = reports[FOCUS]
diag = sd.diagnostics_for_report(report, labels=LABELS, settings=settings)
cleaned = sd.load_cleaned(sd.load_manifest(report["run_id"], settings), settings)

print(f"run {report['run_id']}  |  {report['label_source']}  |  "
      f"holdout: {report['leakage']['restriction']}")
print(diag.header())

### How to read the listings

`show(...)` prints how many pairs are in the cell, then renders the first `N`
of them the way `gold_labeling.ipynb` does: **the two records stacked**, A above
B with their fields aligned column-wise and a blank row between pairs. The eye
compares down a column, not across a row, and the colour says which fields agree
before any of them are read:

| colour | meaning |
|---|---|
| 🟩 green | both records have the field and they agree |
| 🟥 red | both have it and they disagree |
| 🟨 yellow | only one side has it |
| ⬜ grey | neither side has it |

The row label is the PATID, and the leading context columns — `gold_class`,
`error`, the rule that fired, the two model scores, the final route — repeat on
both rows of a pair. `gold_class` is filled solid in its own colour (green
match / amber ambiguous / red non-match) so the adjudicated class is the first
thing you see.

Nothing is capped: the full frame is always returned by the `sd.*` call, and
`show` only limits what is *rendered*. Raise `N`, or slice the frame yourself,
to iterate further down a list. `table(...)` gives the same pairs as a plain
one-row-per-pair frame when you want to sort, filter or export them.

In [ ]:
N = 25   # pairs rendered per listing; raise it to iterate further down a cell


def show(pairs: pd.DataFrame, n: int = None, sort: str | None = None,
         ascending: bool = True, fields=sd.ATTRIBUTE_COLUMNS,
         context=sd.STACKED_CONTEXT):
    """Count the cell, then render its first `n` pairs stacked and colour-coded."""
    n = N if n is None else n
    print(f"{len(pairs):,} pairs in this cell — showing {min(n, len(pairs))}")
    if not len(pairs):
        return pairs
    if sort:
        pairs = pairs.sort_values(sort, ascending=ascending)
    return sd.style_pairs(pairs, cleaned, fields=fields, context=context, limit=n)


def table(pairs: pd.DataFrame, n: int = None, sort: str | None = None,
          ascending: bool = True, fields=sd.ATTRIBUTE_COLUMNS):
    """The same pairs as one flat row each (`<field>_A` next to `<field>_B`) —
    for sorting, filtering and exporting, where the stacked view is awkward."""
    n = N if n is None else n
    print(f"{len(pairs):,} pairs in this cell — showing {min(n, len(pairs))}")
    if not len(pairs):
        return pairs
    if sort:
        pairs = pairs.sort_values(sort, ascending=ascending)
    out = sd.with_attributes(pairs.head(n), cleaned, columns=fields)
    keep = [c for c in (*sd.STACKED_CONTEXT, "PATID_A", "PATID_B") if c in out.columns]
    sides = [c for c in out.columns
             if c.endswith(("_A", "_B")) and c not in ("PATID_A", "PATID_B")]
    return out[keep + sides]

## 2. Where the errors are

One row per error cell, in pipeline order — the index to everything below.
Numbers here match the off-diagonal cells of the matrices in
`end_to_end_eval.ipynb` exactly.

In [ ]:
rows = []
for stage in ("blocking", "gate", "ml_matcher", "clustering"):
    spec = sd.BINARY_STAGES[stage]
    pop = diag.population(stage)
    for kind, label in (("FN", f"{spec.truth_labels[1]} -> {spec.pred_labels[0]}"),
                        ("FP", f"{spec.truth_labels[0]} -> {spec.pred_labels[1]}")):
        rows.append({"stage": stage, "view": "2-class", "cell": label,
                     "pairs": len(sd.binary_errors(diag, stage, kind=kind)),
                     "of population": len(pop)})
for stage in ("rules", "gate", "ml_matcher", "clustering"):
    rows.append({"stage": stage, "view": "3-class routing", "cell": "misrouted (any)",
                 "pairs": len(sd.route_errors(diag, stage)),
                 "of population": len(diag.pairs)})
pd.DataFrame(rows)

## 3. Stage by stage

### 3.1 Blocking — true pairs that were never considered

The unrecoverable ones: no candidate pair was ever emitted, so no rule and no
model ever saw them. Read these for the *evidence* that should have blocked them
— a shared SSN or phone here means a blocking key is missing, while two records
that agree on nothing are simply hard.

In [ ]:
show(sd.binary_errors(diag, "blocking", kind="FN"))

Blocking's false positives (non-matches it emitted) are not errors in any
meaningful sense — over-generation is the design, and the rules, the gate and
the matcher exist to clean up after it. They are listed here only for spot
checks of the block keys.

In [ ]:
show(sd.binary_errors(diag, "blocking", kind="FP"), n=10)

### 3.2 Deterministic rules — wrong rejects and wrong auto-merges

Two cells matter here, and they cost opposite things.

**True pairs the reject rules threw away** — three or more strong-identifier
contradictions fired on a pair that really is one patient. `rules_rule` names
the reject rule; `rules_n_contradictions` says how confident it was.

In [ ]:
show(sd.route_errors(diag, "rules", expected="auto_merge", actual="no_match",
                     population="blocked"))

**Non-matches the rules auto-merged.** The rules' `auto_merge` tier is binding —
clustering unions it — so every pair here is a wrong merge in the shipped
output. `rules_rule` names the rule that confirmed it.

In [ ]:
show(sd.route_errors(diag, "rules", expected="no_match", actual="auto_merge"))

**Ambiguous pairs the rules auto-merged.** Not wrong the same way — the
evidence may well support the merge — but the labeler could not call it, so
these are the merges made on evidence a human declined to act on.


In [ ]:
show(sd.route_errors(diag, "rules", expected="ambiguous", actual="auto_merge"))

Everything else the rules misrouted, including ambiguous pairs they resolved either way:

In [ ]:
show(sd.route_errors(diag, "rules", population="blocked"))

### 3.3 Stage-4.25 gate — dropped pairs and passed noise

**Plausible pairs the gate dropped.** These are unrecoverable: no later stage
ever sees them. Sorted by score ascending, so the ones the model was most
confident about — its worst mistakes — come first.

In [ ]:
show(sd.binary_errors(diag, "gate", kind="FN"), sort="gate_score")

**Confident non-matches the gate passed on.** Cheap by comparison: the matcher
still has to decline them, and they only cost review volume. Sorted by score
descending — the ones it was most sure about.

In [ ]:
show(sd.binary_errors(diag, "gate", kind="FP"), sort="gate_score", ascending=False)

Misrouted after the gate — the cumulative view, every labeled pair:

In [ ]:
show(sd.route_errors(diag, "gate"))

### 3.4 Stage-4.5 matcher — wrong merges and missed merges

**Pairs the matcher auto-merged that are not confident matches.** With
`ml_feeds_clustering` on these are real merges in the output, so this is the
list to read before moving `ml_auto_merge_threshold`. Sorted by score
descending: a wrong merge at 0.99 is a different problem from one at 0.71.

In [ ]:
show(sd.binary_errors(diag, "ml_matcher", kind="FP"), sort="ml_score", ascending=False)

**Confident matches the matcher left in review.** Not lost — a reviewer still
sees them — but each one is a merge a human has to make by hand. Sorted by score
descending, so the near-misses sitting just under the threshold come first.

In [ ]:
show(sd.binary_errors(diag, "ml_matcher", kind="FN"), sort="ml_score", ascending=False)

Misrouted after the matcher:

In [ ]:
show(sd.route_errors(diag, "ml_matcher"))

### 3.5 Clustering — the merges that shipped

**Non-matches that ended up in the same cluster.** The error that actually costs
something: two patients' records merged into one identity.

In [ ]:
show(sd.binary_errors(diag, "clustering", kind="FP"))

**True matches the pipeline never merged** — the end-to-end recall misses:

In [ ]:
show(sd.binary_errors(diag, "clustering", kind="FN"))

**Merged by transitive closure alone.** In the same cluster, but no stage ever
emitted an `auto_merge` edge for the pair — clustering took the connected
component and these came along. No classifier ever scored them, which is why
they are invisible to every per-stage view above.

In [ ]:
transitive = diag.pairs[diag.pairs["clustered"]
                        & (diag.pairs["rules_decision"] != "auto_merge")
                        & ~diag.pairs["ml_auto"]]
show(transitive[~transitive["gold_match"]])

**Final routing errors** — the same three-class view the evaluation notebook
ends on, at pair level. Pull any single cell with `expected=` / `actual=`:

In [ ]:
show(sd.route_errors(diag, "clustering", expected="human_review", actual="auto_merge"))

In [ ]:
show(sd.route_errors(diag, "clustering"))

## 4. Taking a list with you

Uncomment to write one of the frames above to disk. **The file is PHI** — keep
it inside `data/`, which is gitignored in full, and delete it when you are done
with it.

In [ ]:
# errors = sd.with_attributes(sd.binary_errors(diag, "clustering", kind="FP"), cleaned)
# out = settings.project_root / "data" / f"errors_{report['run_id']}_clustering_FP.csv"
# errors.to_csv(out, index=False)
# print(f"wrote {len(errors):,} rows -> {out}")

## 5. Review batches — hand-off to `gold_labeling.ipynb`

At the end of the pipeline a pair can be misrouted in exactly **six** ways — the
off-diagonal of the final 3x3:

| batch | what it is | what it costs |
|---|---|---|
| `match_to_no_match` | a true match the pipeline threw away | unrecoverable |
| `match_to_human_review` | a true match left for a human | slow, not lost |
| `ambiguous_to_no_match` | undecidable evidence dropped without review | unrecoverable |
| `ambiguous_to_auto_merge` | merged on evidence a human declined to call | a merge nobody sanctioned |
| `nonmatch_to_human_review` | noise in the reviewer queue | review volume |
| `nonmatch_to_auto_merge` | two patients merged into one identity | the expensive one |

Each is written as its own CSV so a reviewer adjudicates **one kind of mistake
at a time** — judgement stays calibrated when the question doesn't change every
few rows.

The files carry **pair ids and provenance, not identity fields**:
`gold_labeling.ipynb` rejoins them to its own record frame, so there is one
source of truth for what a record says and a batch can't go stale against it.

> ⚠️ PATIDs are PHI. `data/` is gitignored in full — keep the batches there.

In [ ]:
BATCH_DIR = settings.project_root / "data" / "review_batches" / report["run_id"]

# `stage="clustering"` is the shipped decision. Swap in "rules" / "gate" /
# "ml_matcher" to build batches for where a stage *first* misrouted the pair
# instead — useful when one stage is what you actually want to fix.
index = sd.export_review_batches(diag, BATCH_DIR, stage="clustering")

print(f"wrote {int(index['exported'].sum()):,} pairs to {BATCH_DIR}\n")
index

Large batches are worth sampling rather than truncating — the frame is ordered
by the label file, so its first *n* rows are not a fair look at the error.
`max_per_batch` takes a reproducible random sample:

In [ ]:
# index = sd.export_review_batches(diag, BATCH_DIR, max_per_batch=300, random_state=0)
# index

### Reviewing a batch here first

`review_batches()` returns the same six frames in memory, so a batch can be read
in this notebook before it goes to the labeler:

In [ ]:
batches = sd.review_batches(diag)
show(batches["nonmatch_to_auto_merge"])

### Then, in `gold_labeling.ipynb`

Copy `BATCH_DIR` over to that repo (or point it at the same path) and run its
**"Review batches from the pipeline evaluation"** section: it reads one batch,
matches the ids back to its `pairs` frame irrespective of A/B order, and hands
the subset to `launch_labeler`. Nothing else in that notebook changes — a batch
subset is an ordinary `pairs` slice, exactly like every other subset there.

In [ ]:
print("batch directory to point gold_labeling.ipynb at:")
print(BATCH_DIR)